Loading the parquet file + schema health check

In [11]:
import os
import shutil
import pandas as pd
import pyarrow.parquet as pq
import fsspec
import time
import duckdb
from pathlib import Path


In [4]:
urls = {
    "jan": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet",
    "feb": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet",
    "mar": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet",
    "apr": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-04.parquet",
    "may": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-05.parquet",
    "jun": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet",
    "jul": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet",
    "aug": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-08.parquet",
    "sep": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-09.parquet",
    "oct": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-10.parquet",
    "nov": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet",
    "dec": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-12.parquet",
}

schemas = {}

# schema health check
for name, url in urls.items():
    with fsspec.open(url, "rb") as f:
        pf = pq.ParquetFile(f)
        schemas[name] = pf.schema_arrow
    print(f"\n=== {name} ====")
    print(pf.schema)

ref_month, ref_schema = next(iter(schemas.items()))
bad = [m for m, s in schemas.items() if not s.equals(ref_schema)]
print("All match" if not bad else f" mismatch: {bad}")


=== jan ====
required group field_id=-1 schema {
  optional int32 field_id=-1 VendorID;
  optional int64 field_id=-1 tpep_pickup_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 tpep_dropoff_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 passenger_count;
  optional double field_id=-1 trip_distance;
  optional int64 field_id=-1 RatecodeID;
  optional binary field_id=-1 store_and_fwd_flag (String);
  optional int32 field_id=-1 PULocationID;
  optional int32 field_id=-1 DOLocationID;
  optional int64 field_id=-1 payment_type;
  optional double field_id=-1 fare_amount;
  optional double field_id=-1 extra;
  optional double field_id=-1 mta_tax;
  optional double field_id=-1 tip_amount;
  optional double field_id=-1 tolls_amount;
  optional double field_id=-1 improvement_s

One-time local combined parquet build

In [ ]:
# issue
local_parquet = Path("data/raw/yellow_2025_combined.parquet")
local_monthly_dir = Path("data/raw/monthly_2025")
DOWNLOAD_BUFFER_SIZE = 8 * 1024 * 1024
ROW_GROUP_SIZE = 500000

def materialize_monthly_parquets(urls_dict, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)

    local_paths = []
    t0 = time.perf_counter()

    for url in urls_dict.values():
        filename = url.rsplit("/", 1)[-1]
        target = local_dir / filename

        if not target.exists():
            tmp_target = target.with_suffix(target.suffix + ".tmp")

            try:
                with fsspec.open(url, "rb") as src, open(tmp_target, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=DOWNLOAD_BUFFER_SIZE)
                os.replace(tmp_target, target)
            except Exception:
                tmp_target.unlink(missing_ok=True)
                raise

        local_paths.append(target.as_posix())

    t1 = time.perf_counter()
    print(f"Monthly cache ready in {t1 - t0:.2f}s -> {local_dir}")
    return local_paths

def build_local_combined_parquet(urls_dict, output_path, monthly_dir):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        print(f"Already exists: {output_path}")
        return

    monthly_paths = materialize_monthly_parquets(urls_dict, monthly_dir)

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={max(1, os.cpu_count() or 1)}")
    con.execute("SET preserve_insertion_order = false")

    quoted_paths = ", ".join([f"'{p}'" for p in monthly_paths])

    t0 = time.perf_counter()
    # 500k balances build throughput and downstream scan granularity without large memory spikes during write.
    sql = f"""
    COPY (
        SELECT *
        FROM read_parquet([{quoted_paths}], union_by_name=false)
    )
    TO '{output_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE {ROW_GROUP_SIZE});
    """

    con.execute(sql)
    con.close()

    t1 = time.perf_counter()
    print(f"Built once in {t1 - t0:.2f}s -> {output_path}")

build_local_combined_parquet(urls, local_parquet, local_monthly_dir)


Error: TProtocolException: Invalid data